In [2]:
!pip install tensorflow

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

In [5]:
df = pd.read_csv('fra_cleaned.xls', sep=';', encoding='utf-8', encoding_errors='ignore')
df.columns = df.columns.str.strip()#Gizli boşlukları sil

In [6]:
neccesary_columns = ['Brand','Top','Middle','Base','Gender','mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5','Country','Rating Value','Year']

In [7]:
df[neccesary_columns] = df[neccesary_columns].fillna('')

In [8]:
df['new_csv'] = (
    df['Top'] + ', ' + df['Middle'] + ', ' + df['Base'] + ', ' + 
    df['Gender'] + ', ' + df['mainaccord1'] + ', ' + df['mainaccord2'] + ', ' +
    df['mainaccord3'] + ', ' + df['mainaccord4'] + ', ' + df['mainaccord5'] + ', ' + 
    df['Country'] + ', ' + df['Rating Value'].astype(str) + ', ' + df['Year'].astype(str)
)

In [9]:
vectorizer = CountVectorizer(tokenizer=lambda x: x.split(','), token_pattern=None)
X_rich = vectorizer.fit_transform(df['new_csv']).toarray()
input_shape_rich = X_rich.shape[1]
print("\n success you have more inteligent data right now")
print(f"number of total features to learn by machine : {input_shape_rich}")


 success you have more inteligent data right now
number of total features to learn by machine : 2822


In [10]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
import numpy as np
import joblib

In [11]:
print("1. AI is building now...")
input_layer = Input(shape=(input_shape_rich,))
dense_encoder = Dense(512, activation='relu')(input_layer)
bottleneck = Dense(128, activation='relu')(dense_encoder)


1. AI is building now...


In [12]:
dense_decoder = Dense(512, activation='relu')(bottleneck)
output_layer = Dense(input_shape_rich, activation='sigmoid')(dense_decoder)
autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [13]:
def custom_tokenizer(text):
    return text.split(',')

print("\n2. Final training is starting...")
autoencoder.fit(X_rich, X_rich, epochs=5, batch_size=256)

print("\n3. Extracting the Encoder and generating embeddings...")
encoder = Model(inputs=input_layer, outputs=bottleneck)
perfume_embeddings = encoder.predict(X_rich)

print("\n4. Saving final files for the web application...")
encoder.save('final_encoder.keras')
np.save('perfume_embeddings.npy', perfume_embeddings)

vectorizer.tokenizer = custom_tokenizer
joblib.dump(vectorizer, 'final_vectorizer.pkl')
df.to_csv('final_cleaned_perfumes.csv', index=False, sep=';', encoding='utf-8')

print("\nGRAND FINALE: Machine Learning phase is officially and successfully completed!")


2. Final training is starting...
Epoch 1/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0958
Epoch 2/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0243
Epoch 3/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.0220
Epoch 4/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0189
Epoch 5/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: -0.2931

3. Extracting the Encoder and generating embeddings...
752/752 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

4. Saving final files for the web application...

GRAND FINALE: Machine Learning phase is officially and successfully completed!


In [14]:
from sklearn.metrics.pairwise import cosine_similarity


print("Calculating similarities on enriched AI embeddings...")
enriched_similarity_matrix = cosine_similarity(perfume_embeddings)

# 2. FINAL RECOMMENDATION FUNCTION
def recommend_enriched_perfume(perfume_name, num_recommendations=10, different_brand_only=False):
    try:
        # Find the index and brand of the selected perfume in the dataset
        perfume_index = df[df['Perfume'] == perfume_name].index[0]
        original_brand = df.iloc[perfume_index]['Brand']
    except IndexError:
        return "This perfume was not found in the dataset. Please make sure the name is spelled correctly."
    
    # Get the ENRICHED scores of this perfume with all other perfumes
    similarity_scores = list(enumerate(enriched_similarity_matrix[perfume_index]))
    
    # Sort the scores from highest to lowest
    sorted_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    filtered_indices = []
    
    for i in sorted_scores:
        idx = i[0]
        recommended_brand = df.iloc[idx]['Brand']
        
        # Rule 1: Do not recommend the perfume itself
        if idx == perfume_index:
            continue
            
        # Rule 2: If we requested a "Different Brand" and the brand is the same, skip this perfume
        if different_brand_only and recommended_brand == original_brand:
            continue
            
        # Add the perfume that passed the rules to the list
        filtered_indices.append(idx)
        
        # Break the loop if we reached the desired number of recommendations
        if len(filtered_indices) == num_recommendations:
            break
            
    print(f"\n🤖 Enriched AI recommends the following for lovers of '{perfume_name}' ({original_brand}):")
    print("-" * 90)
    
    # Select the columns we want to display (Gender and Main Accord added!)
    display_columns = ['Perfume', 'Brand', 'Gender', 'mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5', 'Top', 'Middle', 'Base']
    return df.iloc[filtered_indices][display_columns]

# TESTING TIME!
# Let's challenge the algorithm by setting different_brand_only=True
display(recommend_enriched_perfume('le-male-pride-edition', different_brand_only=False))

Calculating similarities on enriched AI embeddings...

🤖 Enriched AI recommends the following for lovers of 'le-male-pride-edition' (jean-paul-gaultier):
------------------------------------------------------------------------------------------


,Perfume,Brand,Gender,mainaccord1,mainaccord2,mainaccord3,mainaccord4,mainaccord5,Top,Middle,Base
759,nude-cashmere,zara,women,citrus,musky,fresh spicy,white floral,aromatic,bergamot,neroli,musk
3463,away-weekend-woman,abercrombie-fitch,women,aromatic,musky,fruity,lavender,green,"white tea, fig, rhubarb","clary sage, lavender, jasmine","musk, ambroxan, tonka bean"
9016,musc-intense,evody-parfums,unisex,musky,powdery,citrus,aromatic,woody,"ambrette (musk mallow), bergamot, elemi",petitgrain,"musk, sandalwood, virginia cedar"
9827,soleil-de-capri-for-gold-apple,montale,unisex,citrus,musky,fresh,white floral,,"kumquat, citruses, grapefruit",white flowers,"musk, spices"
11886,ck-one-collector-bottle-2008,calvin-klein,unisex,fresh spicy,powdery,citrus,warm spicy,tropical,"bergamot, cardamom, pineapple, papaya","tea, nutmeg, jasmine, violet","musk, amber"
306,don-t-tell-jasmine,vilhelm-parfumerie,unisex,white floral,citrus,musky,aromatic,floral,"lemon, kir royal",jasmine,musk
2874,acqua-neroli,clean,unisex,citrus,white floral,musky,fresh spicy,floral,"bergamot, sicilian orange, mandarin orange","neroli, orange blossom, jasmine","musk, ambrette (musk mallow), sandalwood, amber"
10334,verveine-cactus,l-occitane-en-provence,unisex,citrus,green,musky,aromatic,,"verbena, grapefruit, lemon","cactus, aloe vera, verbena, neroli","musk, orange blossom, petitgrain"
2,classique-pride-2023,jean-paul-gaultier,unisex,citrus,white floral,sweet,fresh spicy,musky,"blood orange, yuzu","neroli, orange blossom","musk, white woods"
1745,handsome-black,paris-elysees,men,tropical,fruity,woody,sweet,citrus,"mango, tangerine, aldehydes, lemon","moss, sea notes, amber, violet","cedar, cardamon"
